# Full January 2025 Taxi Benchmark — free Google Colab fallback

This supplements the local Windows Streamlit app. Choose **Runtime → Change runtime type → CPU**; no GPU or paid plan is required. Run cells in order in one runtime so KMeans and MiniBatch share hardware. No results are bundled or simulated in this notebook.

The repository's frozen k, features, scaler, cleaning rules and source hash are reused. Full means **all accepted January rows**, never a smaller substitute. Ordinary estimators receive memory preflight and a 15-minute worker timeout. Insufficient RAM or runtime interruption is an explicit failure, not an invitation to purchase resources.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab; use the README locally.") from exc

REPO_URL = "https://github.com/vighneshvijayjadhav-eng/large-scale_clustering_comparison.git"
REPO_REF = "main"  # For exact reproduction, replace with the delivered commit SHA.
project = Path("/content/large-scale_clustering_comparison")
if not project.exists():
    subprocess.run(["git", "clone", REPO_URL, str(project)], check=True)
os.chdir(project)
subprocess.run(["git", "checkout", REPO_REF], check=True)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

## Isolated supported Python
Colab's default Python can change. This cell installs a disposable Python 3.12 environment inside the clone and the same project dependencies. It does not replace the notebook kernel. Actual library versions and Git revision are recorded in the output.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.12", ".venv"], check=True)
python = str(project / ".venv/bin/python")
subprocess.run(
    [sys.executable, "-m", "uv", "pip", "install", "--python", python, "-e", "."], check=True
)
subprocess.run(
    [
        python,
        "-c",
        "import psutil; print('RAM GiB:', psutil.virtual_memory().total/1024**3, 'available:', psutil.virtual_memory().available/1024**3)",
    ],
    check=True,
)

## Run both full estimators in this runtime
The script automatically downloads the official Parquet unless `data/yellow_tripdata_2025-01.parquet` already exists. You may place that same official file there first; its SHA-256 and accepted count must match the frozen local experiment.

No local model binaries are uploaded: `reports/frozen_preprocessing.json` contains the exact fitted scaler parameters. Preprocessing streams through Polars, the standardized array is float32/memory-mapped, and quality metrics use only 2,000 reproducibly sampled rows. Peak RSS includes the worker process tree.

Set `ALL_SIZES=True` to also run 50k–1m and the batch-size study on the same Colab hardware. The default runs full KMeans, full ordinary MiniBatch and full incremental MiniBatch. A failed full fit is never silently downscaled. If memory is insufficient, restart a free CPU runtime and retry; otherwise retain the failure report.


In [ ]:
ALL_SIZES = False
command = [python, "scripts/run_colab_benchmark.py"]
if ALL_SIZES:
    command.append("--all-sizes")
completed = subprocess.run(command, check=False)
print("Workflow exit code:", completed.returncode)

In [ ]:
import json
import pandas as pd

output = project / "reports/colab"
if (output / "workflow_status.json").exists():
    print((output / "workflow_status.json").read_text())
if (output / "benchmarks.csv").exists():
    display(pd.read_csv(output / "benchmarks.csv"))
if (output / "runtime.json").exists():
    print(json.dumps(json.loads((output / "runtime.json").read_text()), indent=2))

## Export only small reports and import locally
Download the ZIP, then in local PowerShell run:

```powershell
Expand-Archive "$HOME\Downloads\colab_results.zip" -DestinationPath .\reports\colab -Force
.\.venv\Scripts\streamlit run app.py
```

The **Algorithm comparison** page automatically combines `reports/colab/benchmarks.csv` with local observations, validates its frozen experiment fingerprint and labels environment-specific curves. Alternatively upload only the exported `benchmarks.csv` using **Optional Colab benchmark CSV** on that page. That uploader is separate from taxi-trip inference.

Keep CSV/JSON and small HTML reports in Git if you want to publish actual completed Colab results. Do not commit datasets, arrays, model binaries or the ZIP. Cross-environment timing differences include hardware/runtime effects; compare algorithms within the same runtime. There are no Colab measurements until you execute this notebook.


In [ ]:
from google.colab import files

archive = project / "colab_results.zip"
if archive.exists():
    files.download(str(archive))
else:
    print(
        "No export was produced. Inspect the failed cell above; do not substitute invented results."
    )